In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [2]:
import json

from src.config import TOP_K
from src.evaluation import (
    precision_at_k,
    recall_at_k,
    reciprocal_rank,
)
from src.retriever import retrieve

In [3]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd().parent

questions_path = PROJECT_ROOT / "evaluation" / "questions.json"

with open(
    questions_path,
    "r",
    encoding="utf-8",
) as file:
    questions = json.load(file)

questions

[{'question': 'How many days of annual leave do employees receive?',
  'relevant_chunk_ids': [0]},
 {'question': 'How far in advance should employees normally request leave?',
  'relevant_chunk_ids': [0]},
 {'question': 'How many days per week can employees work remotely?',
  'relevant_chunk_ids': [0]},
 {'question': 'What must remote employees do during their working hours?',
  'relevant_chunk_ids': [0]}]

In [4]:
from src.config import RAW_DOCUMENTS_DIR, TOP_K
from src.loader import load_documents
from src.pipeline import build_knowledge_base
from src.retriever import retrieve

documents = load_documents(RAW_DOCUMENTS_DIR)
print(f"Loaded {len(documents)} documents.")

model, index, chunks = build_knowledge_base(documents)
print(f"Created {len(chunks)} chunks.")

/home/prashant/Projects/rag-knowledge-assistant/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 1 documents.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3393.88it/s]


Created 1 chunks.


In [5]:
all_results = []

for item in questions:

    results = retrieve(
        query=item["question"],
        model=model,
        index=index,
        chunks=chunks,
        top_k=TOP_K,
    )

    recall = recall_at_k(
        results,
        item["relevant_chunk_ids"],
    )

    precision = precision_at_k(
        results,
        item["relevant_chunk_ids"],
    )

    mrr = reciprocal_rank(
        results,
        item["relevant_chunk_ids"],
    )

    all_results.append(
        {
            "question": item["question"],
            "recall@3": recall,
            "precision@3": precision,
            "mrr@3": mrr,
        }
    )

all_results

[{'question': 'How many days of annual leave do employees receive?',
  'recall@3': 1.0,
  'precision@3': 1.0,
  'mrr@3': 1.0},
 {'question': 'How far in advance should employees normally request leave?',
  'recall@3': 1.0,
  'precision@3': 1.0,
  'mrr@3': 1.0},
 {'question': 'How many days per week can employees work remotely?',
  'recall@3': 1.0,
  'precision@3': 1.0,
  'mrr@3': 1.0},
 {'question': 'What must remote employees do during their working hours?',
  'recall@3': 1.0,
  'precision@3': 1.0,
  'mrr@3': 1.0}]

In [6]:
for result in all_results:
    print(result)

{'question': 'How many days of annual leave do employees receive?', 'recall@3': 1.0, 'precision@3': 1.0, 'mrr@3': 1.0}
{'question': 'How far in advance should employees normally request leave?', 'recall@3': 1.0, 'precision@3': 1.0, 'mrr@3': 1.0}
{'question': 'How many days per week can employees work remotely?', 'recall@3': 1.0, 'precision@3': 1.0, 'mrr@3': 1.0}
{'question': 'What must remote employees do during their working hours?', 'recall@3': 1.0, 'precision@3': 1.0, 'mrr@3': 1.0}


In [7]:
average_recall = sum(
    result["recall@3"]
    for result in all_results
) / len(all_results)

average_precision = sum(
    result["precision@3"]
    for result in all_results
) / len(all_results)

average_mrr = sum(
    result["mrr@3"]
    for result in all_results
) / len(all_results)

print(f"Average Recall@3: {average_recall:.3f}")
print(f"Average Precision@3: {average_precision:.3f}")
print(f"Average MRR@3: {average_mrr:.3f}")

Average Recall@3: 1.000
Average Precision@3: 1.000
Average MRR@3: 1.000
